# 00 - EDA and leakage-safe split (no labels)
**Cantaloupe flowers - zero-shot pseudo-labels (no ground truth)**

The dataset has images only, no labels. This notebook checks the images, finds the Roboflow copies of each photo and how they were flipped or rotated, links photos of the same scene, and builds a leakage-safe split. It writes `split.json`, `sources.csv` and `copy_transforms.json`, which every other notebook reads. **Run this first.**

Dataset: https://www.kaggle.com/datasets/samirhossain2001/cantaloupe-flower-datasets

**Attach as inputs:** the dataset

## 1 - Setup

In [ ]:
import os
import gc
import re
import json
import glob
import time
import random
import hashlib
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

NB_START = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cv2", cv2.__version__,
      "| device", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| cpu cores", os.cpu_count())

## 2 - Configuration
Every decision taken from the dataset analysis is in this cell.

In [ ]:
@dataclass
class CFG:
    INPUT_ROOTS: list = field(default_factory=lambda: os.environ.get("INPUT_ROOTS", '/kaggle/input').split(os.pathsep))
    OUT_DIR: str = '/kaggle/working/nb0_artifacts' if os.path.isdir('/kaggle/working') else "outputs/nb0_artifacts"
    DATA_DIR: str = '/kaggle/input/datasets/samirhossain2001/cantaloupe-flower-datasets/Cantaloupe flower datasets'  # dataset folder on Kaggle, checked first; otherwise the inputs are searched
    MARKER: str = 'valid/images'  # a folder that exists only in the dataset
    SPLIT_DIRS: dict = field(default_factory=lambda: {'train': 'train/images', 'valid': 'valid/images', 'test': 'test/images'})
    SOURCE_REGEX: str = '^(.+?)_jpg\\.rf\\.'  # first group = the original photo (Roboflow copies share it)
    TIME_REGEX: str = 'IMG(\\d{14})'
    TIME_FORMAT: str = '%Y%m%d%H%M%S'
    GRAY_TOL: float = 2.0  # mean channel difference below this = grayscale copy
    THUMB: int = 64  # thumbnail side used to align copies
    ALIGN_MAD_K: float = 5.0  # copies whose alignment error is above median + k robust sd are not aligned
    BURST_GAP_S: int = 20  # photos this close in time form one group
    NEAR_DUP_HAMMING: int = 4
    EMBED_LINKS: bool = True
    EMBED_SIZE: int = 224
    EMBED_NEG_QUANTILE: float = 99.9  # percentile of different-date similarity used as link threshold
    SPLIT_FRACS: list = field(default_factory=lambda: [0.6, 0.2, 0.2])
    SPLIT_TRIALS: int = 300
    N_QUAL: int = 8
    SMOKE_SOURCES: int = 70  # SMOKE runs keep the first sources in time order
    SEED: int = 42
    NUM_THREADS: int = min(16, 2 * (os.cpu_count() or 2))


cfg = CFG()
SMOKE = False
cfg

## 3 - Helpers
Images are read with OpenCV. The flip / rotation helpers are checked on a toy image first.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def find_artifact(fname, required=True):
    for root in cfg.INPUT_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", fname), recursive=True))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"{fname} not found: attach the output of the notebook that writes it as an input.")
    return None


def find_data_root(rel_path):
    if os.path.exists(os.path.join(cfg.DATA_DIR, rel_path)):
        return cfg.DATA_DIR
    for root in cfg.INPUT_ROOTS:
        for dirpath, _, _ in os.walk(root):
            if os.path.exists(os.path.join(dirpath, rel_path)):
                return dirpath
    raise FileNotFoundError(f"{rel_path} not found: attach the dataset as an input.")


def read_rgb(rel_path):
    a = cv2.imread(os.path.join(DATA_ROOT, rel_path), cv2.IMREAD_COLOR)
    assert a is not None, f"unreadable image: {rel_path}"
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


def dihedral(a, k):
    """k = 0..3: rotate k x 90 degrees counter-clockwise; k = 4..7: the same, then mirror left-right."""
    a = np.rot90(a, k % 4)
    return np.ascontiguousarray(a[:, ::-1] if k >= 4 else a)


def inv_k(k):
    """Inverse of dihedral(., k): rotations undo with the opposite turn, mirrored ones are their own inverse."""
    return k if k >= 4 else (4 - k) % 4


def dihedral_points(xy, k, w, h):
    xy = np.asarray(xy, dtype=np.float64).reshape(-1, 2).copy()
    for _ in range(k % 4):
        xy = np.stack([xy[:, 1], w - xy[:, 0]], axis=1)
        w, h = h, w
    if k >= 4:
        xy[:, 0] = w - xy[:, 0]
    return xy


def dihedral_boxes(boxes, k, w, h):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes
    p1, p2 = dihedral_points(boxes[:, :2], k, w, h), dihedral_points(boxes[:, 2:], k, w, h)
    return np.concatenate([np.minimum(p1, p2), np.maximum(p1, p2)], axis=1)


def box_area(b):
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    a, b = np.asarray(a, dtype=np.float64).reshape(-1, 4), np.asarray(b, dtype=np.float64).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    return inter / np.maximum(box_area(a)[:, None] + box_area(b)[None, :] - inter, 1e-9)


def match_f1_from_iou(iou, n_a, n_b, thr):
    """F1 of a one-to-one matching at IoU >= thr; nan when both sides are empty."""
    if n_a + n_b == 0:
        return float("nan")
    if n_a == 0 or n_b == 0:
        return 0.0
    r, c = linear_sum_assignment(-iou)
    return 2 * int((iou[r, c] >= thr).sum()) / (n_a + n_b)


def match_f1(a, b, thr):
    return match_f1_from_iou(box_iou(a, b), len(a), len(b), thr)


def boot_ci(values, n_boot, seed=0):
    v = np.asarray([x for x in values if not np.isnan(x)], dtype=np.float64)
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    means = np.random.default_rng(seed).choice(v, (n_boot, len(v))).mean(1)
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def hex_rgb(h):
    return tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))


def tag_rect(img, text, x, y, scale):
    """Rectangle of a text tag whose bottom-left corner is (x, y), kept inside the image."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    x = int(min(max(x, 0), img.shape[1] - tw - 6))
    y = int(min(max(y, th + base + 4), img.shape[0]))
    return x, y - th - base - 4, x + tw + 6, y, base


def put_tag(img, text, x, y, rgb, scale):
    x1, y1, x2, y2, base = tag_rect(img, text, x, y, scale)
    cv2.rectangle(img, (x1, y1), (x2, y2), rgb, -1)
    cv2.putText(img, text, (x1 + 3, y2 - base - 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    return x1, y1, x2, y2


def draw_boxes(img, boxes, scores=None, color="#2a78d6", numbers=False, thick=None):
    """Thick box with a dark outline, so it shows on leaves and on bright sky; label on a filled tag at the first
    corner of the box where it does not cover an earlier tag."""
    out = img.copy()
    rgb = hex_rgb(color)
    side = max(out.shape[:2])
    t = thick or max(3, round(side / 160))
    scale = max(0.6, side / 900)
    placed = []
    for i, b in enumerate(np.asarray(boxes).reshape(-1, 4).astype(int)):
        p1, p2 = (int(b[0]), int(b[1])), (int(b[2]), int(b[3]))
        cv2.rectangle(out, p1, p2, (20, 20, 20), t + 3, cv2.LINE_AA)
        cv2.rectangle(out, p1, p2, rgb, t, cv2.LINE_AA)
        label = str(i) if numbers else (f"{scores[i]:.2f}" if scores is not None else "")
        if not label:
            continue
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        h = th + base + 4
        o = (t + 3) // 2
        corners = [(p1[0] - o, p1[1] - o), (p1[0] - o, p2[1] + o + h), (p2[0] - tw - 6 + o, p1[1] - o),
                   (p2[0] - tw - 6 + o, p2[1] + o + h), (p1[0] + o, p1[1] + o + h)]
        free = [c for c in corners if not any(
            r[0] < q[2] and q[0] < r[2] and r[1] < q[3] and q[1] < r[3] for r in [tag_rect(out, label, *c, scale)[:4]] for q in placed)]
        placed.append(put_tag(out, label, *(free[0] if free else corners[0]), rgb, scale))
    return out


def finish(fig, title):
    """Title above the panels (also for tall figures), then show."""
    h = fig.get_figheight()
    fig.suptitle(title, y=1 - 0.12 / h, va="top")
    plt.tight_layout(rect=(0, 0, 1, 1 - 0.5 / h))
    plt.show()


def save_json(obj, name):
    path = os.path.join(cfg.OUT_DIR, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)
    return path


os.makedirs(cfg.OUT_DIR, exist_ok=True)
seed_everything(cfg.SEED)

probe = np.zeros((40, 60), np.uint8)
probe[5:12, 30:50] = 1
for k in range(8):
    ys, xs = np.nonzero(dihedral(probe, k))
    assert np.allclose(dihedral_boxes([[30, 5, 50, 12]], k, 60, 40)[0], [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]), k
    assert np.array_equal(dihedral(dihedral(probe, k), inv_k(k)), probe), k
print("box transform matches the image transform, and each inverse undoes it, for all 8 flips / rotations")

## 4 - Catalogue and integrity
Every file is listed with its source photo (the file-name part before `_jpg.rf.`) and its capture time, read once and hashed.

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
DATA_ROOT = find_data_root(cfg.MARKER)
print("dataset root:", DATA_ROOT)

rows = []
for split, d in cfg.SPLIT_DIRS.items():
    for f in sorted(os.listdir(os.path.join(DATA_ROOT, d))):
        if os.path.splitext(f)[1].lower() not in IMG_EXT:
            continue
        m = re.match(cfg.SOURCE_REGEX, f)
        src = m.group(1) if m else os.path.splitext(f)[0]
        t = re.search(cfg.TIME_REGEX, src)
        rows.append({"id": f"{split}/{f}", "img": f"{d}/{f}", "official_split": split, "source": src,
                     "time": datetime.strptime(t.group(1), cfg.TIME_FORMAT) if t else pd.NaT})
df = pd.DataFrame(rows)
assert df.time.notna().all(), df[df.time.isna()].head()
df["date"] = df.time.dt.strftime("%Y-%m-%d")
if SMOKE:
    keep = df.groupby("source").time.first().sort_values().index[:cfg.SMOKE_SOURCES]
    df = df[df.source.isin(keep)].reset_index(drop=True)
print(f"{len(df)} images from {df.source.nunique()} source photos")
pd.crosstab(df.official_split, df.groupby("source").source.transform("size").rename("copies per source"))

In [ ]:
def dhash64(gray):
    small = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    return int("".join("1" if v else "0" for v in (small[:, 1:] > small[:, :-1]).ravel()), 2)


def scan(rel):
    raw = open(os.path.join(DATA_ROOT, rel), "rb").read()
    img = read_rgb(rel)
    f = img.astype(np.float32)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    thumb = cv2.resize(gray, (cfg.THUMB, cfg.THUMB), interpolation=cv2.INTER_AREA).astype(np.float32)
    return {"height": img.shape[0], "width": img.shape[1], "md5": hashlib.md5(raw).hexdigest(),
            "chroma": float(max(np.abs(f[..., 0] - f[..., 1]).mean(), np.abs(f[..., 1] - f[..., 2]).mean())),
            "mean_r": float(f[..., 0].mean() / 255), "mean_g": float(f[..., 1].mean() / 255), "mean_b": float(f[..., 2].mean() / 255),
            "brightness": float(gray.mean() / 255), "contrast": float(gray.std() / 255),
            "thumb": (thumb - thumb.mean()) / (thumb.std() + 1e-6),
            "dhash": [dhash64(dihedral(gray, k)) for k in range(8)]}


with ThreadPoolExecutor(cfg.NUM_THREADS) as ex:
    scanned = list(tqdm(ex.map(scan, df.img), total=len(df), desc="read + hash"))
THUMBS = {i: s.pop("thumb") for i, s in zip(df.id, scanned)}
df = pd.concat([df, pd.DataFrame(scanned)], axis=1)
df["is_gray"] = df.chroma < cfg.GRAY_TOL
print("all images readable:", len(df))
df.groupby("official_split").agg(images=("id", "size"), sources=("source", "nunique"), grayscale=("is_gray", "sum"))

## 5 - Image statistics
Sizes, grayscale share and colour per official split.

In [ ]:
print("image sizes:", dict(Counter(zip(df.width, df.height))))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
share = df.groupby("official_split").is_gray.mean().reindex(list(cfg.SPLIT_DIRS))
axes[0].bar(share.index, share.values, color="#2a78d6")
axes[0].set_ylim(0, 1)
axes[0].set_title("Share of grayscale images per official split")
for k, split in enumerate(cfg.SPLIT_DIRS):
    part = df[df.official_split == split]
    axes[1].hist(part.brightness, bins=30, alpha=0.55, color=PALETTE[k], label=split)
    axes[2].hist(part[~part.is_gray].mean_g - part[~part.is_gray].mean_b, bins=30, alpha=0.55, color=PALETTE[k], label=split)
axes[1].set_title("Mean brightness per image")
axes[2].set_title("Green minus blue (colour images only)")
for a in axes[1:]:
    a.legend()
plt.tight_layout()
plt.show()

## 6 - Roboflow copies and their transforms
Each source photo has one reference copy (colour when possible). Every other copy is matched to it over the 8 flips / rotations. The model notebooks use these transforms to compare detections across copies.

In [ ]:
def pick_reference(g):
    colour = g[~g.is_gray]
    return (colour if len(colour) else g).sort_values("id").id.iloc[0]


ref_of = {src: pick_reference(g) for src, g in df.groupby("source")}
df["is_ref"] = df.id.isin(set(ref_of.values()))
tf, best, margin = [], [], []
for it in df.itertuples():
    ref = THUMBS[ref_of[it.source]]
    d = np.array([np.abs(dihedral(THUMBS[it.id], k) - ref).mean() for k in range(8)])
    order = np.argsort(d)
    tf.append(int(order[0]))
    best.append(float(d[order[0]]))
    margin.append(float(d[order[1]] - d[order[0]]))
df["tf"], df["align_mae"], df["align_margin"] = tf, best, margin
copies = df[~df.is_ref]
med = copies.align_mae.median()
mad = (copies.align_mae - med).abs().median()
ALIGN_MAX_MAE = float(med + cfg.ALIGN_MAD_K * 1.4826 * mad) if len(copies) else 0.0
df["aligned"] = df.is_ref | ((df.align_mae <= ALIGN_MAX_MAE) & (df.align_margin > df.align_mae))
names = ["identity", "rot90", "rot180", "rot270", "mirror", "rot90+mirror", "rot180+mirror", "rot270+mirror"]
print(f"copies (not reference): {len(copies)} | alignment MAE limit (median + {cfg.ALIGN_MAD_K} robust sd): {ALIGN_MAX_MAE:.3f}")
print("transform that maps each copy onto its reference:", {names[k]: int(v) for k, v in copies.tf.value_counts().items()})
print("copies that could not be aligned (left out of copy consistency):", int((~df.aligned).sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(copies.align_mae, bins=30, color="#2a78d6")
axes[0].axvline(ALIGN_MAX_MAE, color="#eb6834", ls="--", label="limit")
axes[0].set_title("Best alignment error per copy (z-scored 64 px gray)")
axes[0].legend()
axes[1].hist(copies.align_margin, bins=30, color="#1baf7a")
axes[1].set_title("Gap to the second-best transform")
plt.tight_layout()
plt.show()

multi = [s for s, g in df.groupby("source") if len(g) >= 3][:3]
if multi:
    fig, axes = plt.subplots(len(multi), 6, figsize=(18, 3.2 * len(multi)), squeeze=False)
    for r, src in enumerate(multi):
        g = df[df.source == src].sort_values("is_ref", ascending=False).head(3)
        for c, it in enumerate(g.itertuples()):
            img = read_rgb(it.img)
            axes[r, c].imshow(img)
            axes[r, c].set_title(("reference " if it.is_ref else "copy ") + ("gray" if it.is_gray else "colour"), fontsize=9)
            axes[r, 3 + c].imshow(dihedral(img, it.tf))
            axes[r, 3 + c].set_title(f"aligned: {names[it.tf]}", fontsize=9)
        for a in axes[r]:
            a.axis("off")
    plt.suptitle("Roboflow copies of the same photo (left) and after undoing their flip / rotation (right)")
    plt.tight_layout()
    plt.show()

## 7 - Capture time and bursts
Photos taken a few seconds apart show the same plant. They are grouped so that they never end up in different splits.

In [ ]:
src_df = df[df.is_ref].set_index("source")[["id", "img", "time", "date", "official_split", "is_gray"]].sort_values("time")
src_df["n_copies"] = df.groupby("source").size()
gaps = src_df.time.diff().dt.total_seconds().values[1:]
print("gap between consecutive photos (s): quantiles 5/25/50/75/95 =", np.percentile(gaps, [5, 25, 50, 75, 95]).round(1).tolist())
chain = np.concatenate([[0], np.cumsum(gaps > cfg.BURST_GAP_S)])
src_df["time_group"] = chain
sizes = src_df.time_group.value_counts()
print(f"time groups at <= {cfg.BURST_GAP_S} s: {len(sizes)} | largest {sizes.max()} sources "
      f"({sizes.max() / len(src_df):.0%}) | single photos {int((sizes == 1).sum())}")
cross = int(((gaps <= cfg.BURST_GAP_S) & (src_df.official_split.values[1:] != src_df.official_split.values[:-1])).sum())
print(f"consecutive photos <= {cfg.BURST_GAP_S} s apart that sit in different official splits: {cross}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(np.clip(gaps, 1, None), bins=np.logspace(0, np.log10(max(gaps.max(), 2)), 40), color="#2a78d6")
axes[0].set_xscale("log")
axes[0].axvline(cfg.BURST_GAP_S, color="#eb6834", ls="--", label=f"{cfg.BURST_GAP_S} s")
axes[0].set_title("Time between consecutive photos")
axes[0].set_xlabel("seconds")
axes[0].legend()
axes[1].hist(sizes.values, bins=range(1, sizes.max() + 2), color="#1baf7a")
axes[1].set_title("Photos per time group")
plt.tight_layout()
plt.show()
pd.crosstab(src_df.date, src_df.official_split)

## 8 - Duplicates and leakage
Groups are merged when photos are exact or near copies, or when their image embeddings are as similar as almost no pair from different dates.

In [ ]:
parent = {s: s for s in src_df.index}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    parent[find(a)] = find(b)


for g in src_df.groupby("time_group").groups.values():
    for s in list(g)[1:]:
        union(g[0], s)

exact = df.groupby("md5").source.unique()
exact = exact[exact.map(len) > 1]
print("exact copies shared by two source names:", len(exact))
for srcs in exact:
    for s in srcs[1:]:
        union(srcs[0], s)

DHASH = dict(zip(df.id, df.dhash))
ref_hash = {s: DHASH[src_df.id[s]] for s in src_df.index}
near = []
names_src = list(src_df.index)
for i, a in enumerate(names_src):
    ha = ref_hash[a][0]
    for b in names_src[i + 1:]:
        d = min(bin(ha ^ hb).count("1") for hb in ref_hash[b])
        if d <= cfg.NEAR_DUP_HAMMING:
            near.append((a, b, d))
print(f"near copies across source names (dHash, flips / rotations included, <= {cfg.NEAR_DUP_HAMMING} bits):", len(near))
for a, b, _ in near:
    union(a, b)

In [ ]:
net = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
net.fc = torch.nn.Identity()
net = net.to(device).eval()
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)


@torch.no_grad()
def embed(rel):
    gray = cv2.cvtColor(read_rgb(rel), cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (cfg.EMBED_SIZE, cfg.EMBED_SIZE), interpolation=cv2.INTER_AREA)
    x = np.stack([dihedral(gray, k) for k in range(8)]).astype(np.float32) / 255
    x = (np.repeat(x[..., None], 3, axis=-1) - MEAN) / STD
    f = net(torch.from_numpy(x).permute(0, 3, 1, 2).to(device)).mean(0)
    return torch.nn.functional.normalize(f, dim=0).cpu().numpy()


E = np.stack([embed(r) for r in tqdm(src_df.img, desc="embeddings (gray, 8 flips / rotations averaged)")])
S = E @ E.T
dates = src_df.date.values
iu = np.triu_indices(len(E), 1)
cross_date = dates[iu[0]] != dates[iu[1]]
if cross_date.any():
    EMBED_THRESHOLD = float(np.percentile(S[iu][cross_date], cfg.EMBED_NEG_QUANTILE))
    same_date = ~cross_date & (src_df.time_group.values[iu[0]] != src_df.time_group.values[iu[1]])
    hits = [(names_src[i], names_src[j], float(S[i, j])) for i, j in zip(iu[0][same_date], iu[1][same_date]) if S[i, j] >= EMBED_THRESHOLD]
else:
    EMBED_THRESHOLD, hits = float("nan"), []
print(f"threshold = {cfg.EMBED_NEG_QUANTILE} percentile of similarity between photos of different dates: {EMBED_THRESHOLD:.3f}")
print("same-date links between different time groups:", len(hits))
for a, b, _ in hits:
    union(a, b)

if cross_date.any():
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(S[iu][cross_date], bins=60, alpha=0.6, density=True, color="#8a8984", label="different dates")
    ax.hist(S[iu][~cross_date], bins=60, alpha=0.6, density=True, color="#2a78d6", label="same date")
    ax.axvline(EMBED_THRESHOLD, color="#eb6834", ls="--", label="link threshold")
    ax.set_title("Embedding similarity between source photos")
    ax.legend()
    plt.show()
top = sorted(hits, key=lambda h: -h[2])[:4]
if top:
    fig, axes = plt.subplots(2, len(top), figsize=(4 * len(top), 8), squeeze=False)
    for c, (a, b, s) in enumerate(top):
        axes[0, c].imshow(read_rgb(src_df.img[a]))
        axes[0, c].set_title(f"{a}\n{src_df.official_split[a]}", fontsize=8)
        axes[1, c].imshow(read_rgb(src_df.img[b]))
        axes[1, c].set_title(f"{b}\n{src_df.official_split[b]} | sim {s:.3f}", fontsize=8)
        for a_ in axes[:, c]:
            a_.axis("off")
    plt.suptitle("Strongest embedding links between different time groups (merged into one group)")
    plt.tight_layout()
    plt.show()

In [ ]:
src_df["group"] = [find(s) for s in src_df.index]
codes = {g: k for k, g in enumerate(sorted(src_df.group.unique()))}
src_df["group"] = src_df.group.map(codes)
spread = src_df.groupby("group").official_split.nunique()
leaky = spread[spread > 1].index
print(f"final groups: {src_df.group.nunique()} | groups split across official splits: {len(leaky)} "
      f"({src_df.group.isin(leaky).sum()} of {len(src_df)} source photos)")
print("the official split leaks near-identical scenes, so the data is re-split by group" if len(leaky) else "no cross-split group found")

## 9 - Leakage-safe split
Whole groups go to train / validation / test, balanced per capture date. The most even assignment over several seeds is kept.

In [ ]:
SPLITS = ["train", "validation", "test"]
target = np.array(cfg.SPLIT_FRACS, dtype=np.float64)
groups = src_df.groupby("group").agg(n=("id", "size"), date=("date", "first"))
all_dates = sorted(groups.date.unique())
best = None
for trial in range(cfg.SPLIT_TRIALS):
    rng = np.random.default_rng(cfg.SEED + trial)
    counts = {d: np.zeros(3) for d in all_dates}
    assign = {}
    for g in rng.permutation(groups.index.values):
        d, n = groups.at[g, "date"], groups.at[g, "n"]
        need = target * (counts[d].sum() + n) - counts[d]
        k = int(np.argmax(need + 1e-6 * rng.random(3)))
        assign[g] = k
        counts[d][k] += n
    tot = sum(counts.values())
    score = np.abs(tot / tot.sum() - target).sum() + np.mean([np.abs(c / c.sum() - target).sum() for c in counts.values()])
    if best is None or score < best[0]:
        best = (score, assign)
src_df["split"] = src_df.group.map(lambda g: SPLITS[best[1][g]])
df["split"] = df.source.map(src_df.split)
df["group"] = df.source.map(src_df.group)
assert (df.groupby("group").split.nunique() == 1).all()
for s in SPLITS:
    assert (src_df.split == s).any(), f"split {s} is empty"
print("source photos per split and date:")
display(pd.crosstab(src_df.date, src_df.split, margins=True)[SPLITS + ["All"]])
print("official split (rows) -> new split (columns), source photos:")
display(pd.crosstab(src_df.official_split, src_df.split)[SPLITS])
df.groupby("split").agg(images=("id", "size"), sources=("source", "nunique"), grayscale=("is_gray", "sum"),
                        multi_copy_sources=("source", lambda s: int((s.value_counts() > 1).sum()))).reindex(SPLITS)

## 10 - Examples
Random photos per new split and the fixed test photos every model notebook shows.

In [ ]:
rng = np.random.default_rng(cfg.SEED)
fig, axes = plt.subplots(3, 6, figsize=(18, 9.5))
for r, s in enumerate(SPLITS):
    pool = src_df[src_df.split == s]
    pick = pool.iloc[rng.choice(len(pool), min(6, len(pool)), replace=False)]
    for c in range(6):
        axes[r, c].axis("off")
        if c < len(pick):
            axes[r, c].imshow(read_rgb(pick.img.iloc[c]))
            axes[r, c].set_title(f"{s} | {pick.index[c]}", fontsize=8)
plt.suptitle("Reference copy of random source photos per new split")
plt.tight_layout()
plt.show()

multi_test = [s for s in src_df[src_df.split == "test"].index if src_df.n_copies[s] > 1]
single_test = [s for s in src_df[src_df.split == "test"].index if src_df.n_copies[s] == 1]
QUAL = sorted(rng.choice(multi_test, min(cfg.N_QUAL // 2, len(multi_test)), replace=False).tolist()
              + rng.choice(single_test, min(cfg.N_QUAL - cfg.N_QUAL // 2, len(single_test)), replace=False).tolist())
print("fixed qualitative sources (test):", QUAL)

## 11 - Save the split
Items store paths relative to the dataset root. No augmentation check: nothing is trained in this project.

In [ ]:
cols = ["id", "img", "source", "group", "date", "official_split", "is_gray", "is_ref", "tf", "aligned", "width", "height"]
items = {s: [{k: (v.item() if hasattr(v, "item") else v) for k, v in r.items()}
             for r in df[df.split == s].sort_values("id")[cols].to_dict("records")] for s in SPLITS}
split_id = hashlib.md5(json.dumps(sorted((i, s) for s in SPLITS for i in df[df.split == s].id)).encode()).hexdigest()[:12]
meta = {"split_id": split_id, "task": "zero_shot_detection", "items": items, "qual_sources": QUAL,
        "burst_gap_s": cfg.BURST_GAP_S, "embed_threshold": EMBED_THRESHOLD, "align_max_mae": ALIGN_MAX_MAE,
        "split_fracs": cfg.SPLIT_FRACS, "transform_names": names, "smoke": SMOKE}
print("wrote", save_json(meta, "split.json"))
src_df.reset_index().drop(columns=["id"]).to_csv(os.path.join(cfg.OUT_DIR, "sources.csv"), index=False)
save_json({i: {"tf": int(t), "mae": float(m), "aligned": bool(a)} for i, t, m, a in zip(df.id, df.tf, df.align_mae, df.aligned)},
          "copy_transforms.json")
print(f"split_id {split_id} | {len(df)} images | outputs in {cfg.OUT_DIR}: split.json, sources.csv, copy_transforms.json")
print(f"total time {(time.time() - NB_START) / 60:.1f} min")